# 01 — Data Understanding

This notebook profiles the prepared financial statement fraud dataset before any modelling. The first pass uses the documented **time-split, fold 1** train and test CSVs.

**Important:** Keep train and test separate. Do not use the test set to make modelling decisions. This notebook is for initial schema and data-quality inspection only.

Dataset source: [WaguyMZ / Financial Statement Fraud Detection Dataset](https://huggingface.co/datasets/WaguyMZ/Financial_statements_fraud_dataset).

Before proceeding, review [`docs/DATASET_REVIEW.md`](../docs/DATASET_REVIEW.md).

## 1. Environment setup

From the repository root, install the project dependencies:

```bash
python -m pip install -r requirements.txt
```

If the Hugging Face filesystem protocol is unavailable in your environment, update/install `huggingface_hub` and `fsspec` as documented by their current installation instructions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

RANDOM_STATE = 42
SPLIT_STRATEGY = 'time_splitting'
FOLD = 1
BASE_URI = 'hf://datasets/WaguyMZ/Financial_statements_fraud_dataset/FINAL_DATASETS'

print('Pandas version:', pd.__version__)
print('Dataset split:', SPLIT_STRATEGY, '| Fold:', FOLD)

## 2. Load one explicit split

The source has different file schemas across some configurations. Load only the selected train/test pair instead of asking a dataset loader to combine every CSV.

In [ ]:
train_uri = f'{BASE_URI}/{SPLIT_STRATEGY}/fold_{FOLD}/train.csv'
test_uri = f'{BASE_URI}/{SPLIT_STRATEGY}/fold_{FOLD}/test.csv'

train = pd.read_csv(train_uri)
test = pd.read_csv(test_uri)

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
display(train.head())

In [ ]:
# Confirm that the selected train and test files have compatible columns.
train_columns = set(train.columns)
test_columns = set(test.columns)

print('Columns only in train:', sorted(train_columns - test_columns))
print('Columns only in test: ', sorted(test_columns - train_columns))
print('Same column set:', train_columns == test_columns)

## 3. Understand the schema

Check data types and non-null counts. We will not assume that a column name or a precomputed feature is reliable until its meaning is confirmed against the dataset documentation.

In [ ]:
train.info()

display(
    pd.DataFrame({
        'column': train.columns,
        'dtype': train.dtypes.astype(str).values,
        'non_null': train.notna().sum().values,
        'missing': train.isna().sum().values,
        'missing_pct': (train.isna().mean().values * 100).round(2),
        'unique_values': train.nunique(dropna=True).values,
    }).sort_values('missing_pct', ascending=False)
)

## 4. Data-quality checks

These checks are descriptive only. A duplicate row is not automatically invalid: the correct key needs to be established from the source schema.

In [ ]:
print('Exact duplicate rows in train:', train.duplicated().sum())
print('Exact duplicate rows in test: ', test.duplicated().sum())

missing_summary = (
    train.isna().mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .to_frame()
)
display(missing_summary.head(30))

## 5. Target and class balance

The prepared dataset documents an `is_fraud` target. Confirm that this field exists and inspect the class balance separately in train and test. Do not rebalance or resample the test set.

In [ ]:
TARGET = 'is_fraud'

if TARGET in train.columns and TARGET in test.columns:
    train_target = train[TARGET].value_counts(dropna=False).rename('count').to_frame()
    train_target['share_pct'] = (train_target['count'] / len(train) * 100).round(2)
    test_target = test[TARGET].value_counts(dropna=False).rename('count').to_frame()
    test_target['share_pct'] = (test_target['count'] / len(test) * 100).round(2)
    print('TRAIN target distribution')
    display(train_target)
    print('TEST target distribution')
    display(test_target)
else:
    print(f'Expected target column {TARGET!r} is missing. Review the selected file schema before proceeding.')

## 6. Time and company coverage

Inspect key identifiers when present. We use these checks to understand coverage and to guard against company/time leakage in later modelling.

In [ ]:
for column in ['year', 'fiscal_quarter', 'quarter', 'period', 'cik', 'sic']:
    if column in train.columns:
        print(f'--- TRAIN: {column} ---')
        print('Unique values:', train[column].nunique(dropna=True))
        print('Missing values:', train[column].isna().sum())
        if column in ['year', 'sic']:
            display(train[column].value_counts(dropna=False).head(15))

if 'year' in train.columns and 'year' in test.columns:
    print('Train year range:', train['year'].min(), 'to', train['year'].max())
    print('Test year range: ', test['year'].min(), 'to', test['year'].max())

if 'cik' in train.columns and 'cik' in test.columns:
    train_companies = set(train['cik'].dropna().unique())
    test_companies = set(test['cik'].dropna().unique())
    print('Companies in both train and test:', len(train_companies & test_companies))

## 7. Initial findings — complete after running

Record the actual results here after running the notebook:

- Train and test row/column counts:
- Confirmed target column and class distribution:
- Missingness that needs investigation:
- Duplicate rows and candidate business key:
- Time coverage and company overlap:
- Suspicious or ambiguous columns requiring documentation:
- Any loading or schema issues:

**Do not write conclusions before running the cells.** The next notebook will perform cleaning only after we review these findings.

## 8. Next steps

After this notebook is run and reviewed:
1. Create a data dictionary for the selected split.
2. Identify target leakage and duplicated/derived fields.
3. Decide which financial variables support meaningful descriptive analysis.
4. Plan the cleaning and validation rules before modifying any records.